In [ ]:
from functools import lru_cache
from pathlib import Path
import csv
import gzip
import json
import pickle
import random
import time
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
from tqdm.auto import tqdm


def load_pickle(path: Path):
    with path.open("rb") as handle:
        return pickle.load(handle)


def save_pickle_highest_protocol(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)


DATA_ROOT = Path("<DATA_ROOT>")
# Folder with qidpidtriples.train.full.2.tsv.

PREP_MANIFEST_PATH = Path("<PREP_MANIFEST_PATH>")
# Path to prep_manifest.json produced by 00_prepare_data_cache.ipynb.

OUTPUT_ROOT = Path("<OUTPUT_ROOT>")
# Output folder where trm_reranker_mvp/run_data_cache will be created.

RUN_PROFILE = "smoke"
SEED = 13
TRAIN_TRIPLES_SAMPLE = None
DEV_EVAL_MODE = "quick_count"
DEV_EVAL_QUERY_COUNT = 700
DEV_EVAL_FRACTION = 0.05
DEV_EVAL_SEED = 13
RUN_FINAL_FULL_DEV = False
FORCE_REBUILD_RUN_DATA = False

PROFILE_SETTINGS = {
    "smoke": {
        "train_triples_sample": 100_000,
        "run_final_full_dev": True,
        "dev_eval_mode": "quick_count",
        "dev_eval_query_count": 700,
        "dev_eval_seed": 13,
    },
    "full": {
        "train_triples_sample": 20_000_000,
        "run_final_full_dev": True,
        "dev_eval_mode": "quick_count",
        "dev_eval_query_count": 700,
        "dev_eval_seed": 13,
    },
    "full_all_dev": {
        "train_triples_sample": 1_000_000,
        "run_final_full_dev": True,
        "dev_eval_mode": "full",
        "dev_eval_seed": 13,
    },
}
if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unsupported RUN_PROFILE={RUN_PROFILE!r}. Expected one of {sorted(PROFILE_SETTINGS)}")
PROFILE = dict(PROFILE_SETTINGS[RUN_PROFILE])

if TRAIN_TRIPLES_SAMPLE is None:
    TRAIN_TRIPLES_SAMPLE = int(PROFILE["train_triples_sample"])
else:
    TRAIN_TRIPLES_SAMPLE = int(TRAIN_TRIPLES_SAMPLE)
DEV_EVAL_MODE = str(PROFILE.get("dev_eval_mode", DEV_EVAL_MODE))
DEV_EVAL_QUERY_COUNT = PROFILE.get("dev_eval_query_count", DEV_EVAL_QUERY_COUNT)
DEV_EVAL_QUERY_COUNT = int(DEV_EVAL_QUERY_COUNT) if DEV_EVAL_QUERY_COUNT is not None else None
DEV_EVAL_FRACTION = PROFILE.get("dev_eval_fraction", DEV_EVAL_FRACTION)
DEV_EVAL_FRACTION = float(DEV_EVAL_FRACTION) if DEV_EVAL_FRACTION is not None else None
DEV_EVAL_SEED = int(PROFILE.get("dev_eval_seed", DEV_EVAL_SEED))
RUN_FINAL_FULL_DEV = bool(PROFILE.get("run_final_full_dev", RUN_FINAL_FULL_DEV))
FORCE_REBUILD_RUN_DATA = bool(FORCE_REBUILD_RUN_DATA)

if DEV_EVAL_MODE not in {"quick_count", "quick_fraction", "full"}:
    raise ValueError(f"Unsupported DEV_EVAL_MODE={DEV_EVAL_MODE!r}")
if DEV_EVAL_MODE == "quick_count" and (DEV_EVAL_QUERY_COUNT is None or DEV_EVAL_QUERY_COUNT <= 0):
    raise ValueError(f"DEV_EVAL_QUERY_COUNT must be > 0 for quick_count mode, got {DEV_EVAL_QUERY_COUNT!r}")
if DEV_EVAL_MODE == "quick_fraction" and DEV_EVAL_FRACTION is None:
    raise ValueError("DEV_EVAL_FRACTION is required for quick_fraction mode")

REPO_ROOT = Path.cwd().resolve()


In [ ]:
TRAIN_TRIPLES_FILENAME = "qidpidtriples.train.full.2.tsv"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def open_text_auto(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8", errors="replace", newline="")
    return path.open("r", encoding="utf-8", errors="replace", newline="")


def sanitize_tag_fragment(value: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "-" for ch in str(value)).strip("-") or "value"


def resolve_relative_or_absolute_path(value: str, base_dir: Path) -> Path:
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path.resolve()
    return (base_dir / path).resolve()


def maybe_relative_to(path: Path, root: Optional[Path]) -> str:
    resolved_path = path.resolve()
    if root is not None:
        resolved_root = root.resolve()
        try:
            return str(resolved_path.relative_to(resolved_root))
        except ValueError:
            pass
    return str(resolved_path)


def get_manifest_mapping(manifest: dict, section_name: str) -> Dict[str, object]:
    value = manifest.get(section_name) or {}
    if not isinstance(value, dict):
        return {}
    return value


def manifest_value_candidates(primary_key: str, aliases: Optional[List[str]] = None) -> List[str]:
    keys = [primary_key]
    if aliases:
        keys.extend(aliases)
    unique: List[str] = []
    seen = set()
    for key in keys:
        if not key or key in seen:
            continue
        seen.add(key)
        unique.append(key)
    return unique


def get_manifest_mapping_value(mapping: Dict[str, object], keys: List[str]):
    for key in keys:
        value = mapping.get(key)
        if value:
            return value, key
    return None, None


def resolve_artifact_path(manifest: dict, key: str, artifact_dir: Path, aliases: Optional[List[str]] = None) -> Path:
    artifacts = get_manifest_mapping(manifest, "artifacts")
    value, _ = get_manifest_mapping_value(artifacts, manifest_value_candidates(key, aliases))
    if value is None:
        raise KeyError(
            f"Prep manifest is missing required artifact key {key!r}. Available artifact keys: {sorted(artifacts)}"
        )
    return resolve_relative_or_absolute_path(str(value), artifact_dir)


def validate_prep_manifest(manifest: dict) -> None:
    schema_version = int(manifest.get("schema_version", 0))
    if schema_version < 4:
        raise ValueError(
            "Prep manifest schema_version is too old for the dataset-level cache workflow. "
            "Re-run 00_prepare_data_cache.ipynb to build the sharded full-collection passage cache."
        )

    required_artifact_keys = {
        "train_query_tokens_pkl": [],
        "dev_query_tokens_pkl": [],
        "passage_token_shards_dir": ["passage_tokens_shards_dir"],
        "passage_token_shards_index_json": ["passage_tokens_store_index_json"],
        "passage_token_store_stats_json": ["passage_tokens_store_stats_json"],
        "dev_candidates_pkl": [],
        "dev_qrels_pkl": [],
    }
    artifacts = get_manifest_mapping(manifest, "artifacts")
    missing_artifacts = [
        key
        for key, aliases in required_artifact_keys.items()
        if get_manifest_mapping_value(artifacts, manifest_value_candidates(key, aliases))[0] is None
    ]
    if missing_artifacts:
        raise ValueError(
            "Prep manifest is missing required dataset-level artifacts. "
            f"Re-run 00_prepare_data_cache.ipynb. Missing keys: {missing_artifacts}. "
            f"Available artifact keys: {sorted(artifacts)}"
        )


def load_triples_tsv(path: Path) -> List[Tuple[int, int, int]]:
    triples: List[Tuple[int, int, int]] = []
    with open_text_auto(path) as handle:
        reader = csv.reader(handle, delimiter="\t")
        for row in tqdm(reader, desc=f"Load {path.name}"):
            if len(row) >= 3:
                triples.append((int(row[0]), int(row[1]), int(row[2])))
    return triples


def write_triples_tsv(path: Path, triples: Iterable[Tuple[int, int, int]]) -> None:
    triples = list(triples)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t")
        for triple in tqdm(triples, total=len(triples), desc=f"Write {path.name}"):
            writer.writerow(triple)


def reservoir_sample_triples(path: Path, sample_size: int, seed: int) -> Tuple[List[Tuple[int, int, int]], int]:
    if sample_size <= 0:
        raise ValueError(f"TRAIN_TRIPLES_SAMPLE must be > 0, got {sample_size}")
    rng = random.Random(seed)
    sample: List[Tuple[int, int, int]] = []
    seen = 0
    with open_text_auto(path) as handle:
        reader = csv.reader(handle, delimiter="\t")
        for row in tqdm(reader, desc="Sample train triples"):
            if len(row) < 3:
                continue
            triple = (int(row[0]), int(row[1]), int(row[2]))
            seen += 1
            if len(sample) < sample_size:
                sample.append(triple)
            else:
                idx = rng.randrange(seen)
                if idx < sample_size:
                    sample[idx] = triple
    return sample, seen


def build_run_train_subset(triples_path: Path, output_path: Path, sample_size: int, seed: int, force: bool = False) -> List[Tuple[int, int, int]]:
    if output_path.exists() and not force:
        print(f"Reusing sampled train triples from {output_path}")
        return load_triples_tsv(output_path)
    sampled_triples, seen = reservoir_sample_triples(triples_path, sample_size, seed)
    write_triples_tsv(output_path, sampled_triples)
    print(f"Wrote {len(sampled_triples)} sampled triples out of {seen:,} source rows to {output_path}")
    return sampled_triples


def collect_train_subset_ids(triples: List[Tuple[int, int, int]]) -> Tuple[List[int], List[int]]:
    unique_qids = set()
    unique_pids = set()
    for qid, pos_pid, neg_pid in tqdm(triples, total=len(triples), desc="Collect train qids/pids"):
        unique_qids.add(int(qid))
        unique_pids.add(int(pos_pid))
        unique_pids.add(int(neg_pid))
    return sorted(unique_qids), sorted(unique_pids)


def select_query_token_subset(full_query_token_map: Dict[int, List[int]], qids: List[int], desc: str) -> Dict[int, List[int]]:
    subset: Dict[int, List[int]] = {}
    missing_qids: List[int] = []
    for qid in tqdm(qids, total=len(qids), desc=desc):
        qid = int(qid)
        if qid not in full_query_token_map:
            missing_qids.append(qid)
            continue
        subset[qid] = full_query_token_map[qid]
    if missing_qids:
        preview = missing_qids[:5]
        raise KeyError(f"Missing cached query tokens for {len(missing_qids)} qids, e.g. {preview}")
    return subset


def load_passage_token_shard_index(index_path: Path) -> dict:
    return json.loads(index_path.read_text())


def build_passage_token_subset_loader(
    index_path: Path,
    artifact_dir: Path,
    shards_dir_path: Optional[Path] = None,
    shard_cache_size: int = 8,
):
    index = load_passage_token_shard_index(index_path)
    if index.get("format") != "sharded_flat_token_arrays_v1":
        raise ValueError(
            f"Unsupported passage token store format: {index.get('format')!r}. "
            "This notebook expects a sharded flat token store described by passage_token_shards_index_json."
        )

    shard_entries = index.get("shards") or []
    if not shard_entries:
        raise ValueError(f"Passage token shard index is missing shard entries: {index_path}")

    shard_size = int(index["shard_size"])
    shards_by_id = {int(entry["shard_id"]): entry for entry in shard_entries}
    resolved_shards_dir = Path(shards_dir_path).resolve() if shards_dir_path is not None else None

    def resolve_shard_path(shard_entry: dict) -> Path:
        raw_path_value = shard_entry.get("path")
        if raw_path_value:
            return resolve_relative_or_absolute_path(str(raw_path_value), artifact_dir)

        filename = shard_entry.get("filename") or shard_entry.get("file_name") or shard_entry.get("basename") or shard_entry.get("name")
        if filename and resolved_shards_dir is not None:
            return resolved_shards_dir / str(filename)

        raise FileNotFoundError(
            "Passage token shard entry does not provide a path or filename: "
            f"shard_id={shard_entry.get('shard_id')}"
        )

    @lru_cache(maxsize=shard_cache_size)
    def load_shard(shard_id: int):
        shard_entry = shards_by_id.get(int(shard_id))
        if shard_entry is None:
            raise KeyError(f"No passage token shard found for shard_id={shard_id}")
        shard_path = resolve_shard_path(shard_entry)
        with np.load(shard_path, allow_pickle=False) as shard_data:
            shard_pids = shard_data["pid"]
            shard_offsets = shard_data["offsets"]
            shard_token_ids = shard_data["token_ids"]
        pid_lookup = {int(pid): idx for idx, pid in enumerate(shard_pids.tolist())}
        return {
            "pid": shard_pids,
            "offsets": shard_offsets,
            "token_ids": shard_token_ids,
            "pid_lookup": pid_lookup,
        }

    def get_passage_tokens(pid: int) -> List[int]:
        shard = load_shard(int(pid) // shard_size)
        pid_idx = shard["pid_lookup"].get(int(pid))
        if pid_idx is None:
            raise KeyError(f"Missing cached passage tokens for pid={pid}")
        start = int(shard["offsets"][pid_idx])
        end = int(shard["offsets"][pid_idx + 1])
        return shard["token_ids"][start:end].tolist()

    def load_passage_tokens_for_subset(pid_list: Iterable[int]) -> Dict[int, List[int]]:
        unique_pids = sorted({int(pid) for pid in pid_list})
        pids_by_shard: Dict[int, List[int]] = {}
        for pid in unique_pids:
            pids_by_shard.setdefault(int(pid) // shard_size, []).append(int(pid))

        subset: Dict[int, List[int]] = {}
        with tqdm(total=len(unique_pids), desc="Load cached train passages") as pbar:
            for shard_id in sorted(pids_by_shard):
                shard = load_shard(shard_id)
                for pid in pids_by_shard[shard_id]:
                    pid_idx = shard["pid_lookup"].get(pid)
                    if pid_idx is None:
                        raise KeyError(f"Missing cached passage tokens for pid={pid}")
                    start = int(shard["offsets"][pid_idx])
                    end = int(shard["offsets"][pid_idx + 1])
                    subset[pid] = shard["token_ids"][start:end].tolist()
                    pbar.update(1)
        return subset

    return index, get_passage_tokens, load_passage_tokens_for_subset


def normalize_dev_eval_fraction(fraction: float) -> float:
    if fraction is None:
        raise ValueError("DEV_EVAL_FRACTION is required for quick_fraction mode")
    normalized_fraction = float(fraction)
    if not (0.0 < normalized_fraction <= 1.0):
        raise ValueError(f"DEV_EVAL_FRACTION must be in (0, 1], got {fraction!r}")
    return normalized_fraction


def normalize_dev_eval_query_count(query_count: int) -> int:
    if query_count is None:
        raise ValueError("DEV_EVAL_QUERY_COUNT is required for quick_count mode")
    normalized_count = int(query_count)
    if normalized_count <= 0:
        raise ValueError(f"DEV_EVAL_QUERY_COUNT must be > 0, got {query_count!r}")
    return normalized_count


def format_fraction_label(fraction: float) -> str:
    percent = normalize_dev_eval_fraction(fraction) * 100.0
    if abs(percent - round(percent)) < 1e-9:
        return str(int(round(percent)))
    return sanitize_tag_fragment(f"{percent:g}".replace(".", "p"))


def format_dev_eval_mode_label(mode: str, fraction: Optional[float] = None, query_count: Optional[int] = None) -> str:
    if mode == "quick_count":
        return f"quick_count_{normalize_dev_eval_query_count(query_count)}"
    if mode == "quick_fraction":
        return f"quick_fraction_{format_fraction_label(fraction)}"
    if mode == "full":
        return "full"
    raise ValueError(f"Unsupported DEV_EVAL_MODE={mode!r}")


def cast_like(source_values, values):
    if isinstance(source_values, np.ndarray):
        return np.asarray(list(values), dtype=source_values.dtype)
    return list(values)


def get_valid_dev_qids(candidates_artifact, qrels: Dict[int, set]) -> List[int]:
    valid_qids = [int(qid) for qid in candidates_artifact["qid_order"] if int(qid) in qrels]
    if not valid_qids:
        raise ValueError("Cannot build a dev subset because there are no overlapping qids between candidates and qrels.")
    return valid_qids


def build_dev_subset_for_qids(candidates_artifact, qrels: Dict[int, set], selected_qids: List[int]):
    qid_order = candidates_artifact["qid_order"]
    qid_offsets = candidates_artifact["qid_offsets"]
    pid_values = candidates_artifact["pid"]
    bm25_ranks = candidates_artifact["bm25_rank"]
    qid_to_index = {int(qid): idx for idx, qid in enumerate(qid_order)}

    subset_qid_order: List[int] = []
    subset_qid_offsets = [0]
    subset_pid_values = []
    subset_bm25_ranks = []
    subset_qrels: Dict[int, set] = {}

    for qid in selected_qids:
        qid = int(qid)
        idx = qid_to_index[qid]
        start = int(qid_offsets[idx])
        end = int(qid_offsets[idx + 1])
        subset_qid_order.append(qid)
        subset_pid_values.extend(pid_values[start:end])
        subset_bm25_ranks.extend(bm25_ranks[start:end])
        subset_qid_offsets.append(len(subset_pid_values))
        subset_qrels[qid] = set(qrels[qid])

    subset_candidates_artifact = {
        "format": candidates_artifact.get("format", "grouped_arrays_v1"),
        "qid_order": cast_like(qid_order, subset_qid_order),
        "qid_offsets": cast_like(qid_offsets, subset_qid_offsets),
        "pid": cast_like(pid_values, subset_pid_values),
        "bm25_rank": cast_like(bm25_ranks, subset_bm25_ranks),
    }
    return subset_candidates_artifact, subset_qrels


def build_count_dev_subset(candidates_artifact, qrels: Dict[int, set], query_count: int, seed: int):
    qid_order = candidates_artifact["qid_order"]
    valid_qids = get_valid_dev_qids(candidates_artifact, qrels)
    target_count = min(len(valid_qids), normalize_dev_eval_query_count(query_count))
    if target_count >= len(valid_qids):
        selected_qids = list(valid_qids)
    else:
        rng = random.Random(int(seed))
        sampled_qids = set(rng.sample(valid_qids, target_count))
        selected_qids = [int(qid) for qid in qid_order if int(qid) in sampled_qids and int(qid) in qrels]
    return build_dev_subset_for_qids(candidates_artifact, qrels, selected_qids)


def build_fractional_dev_subset(candidates_artifact, qrels: Dict[int, set], fraction: float, seed: int):
    qid_order = candidates_artifact["qid_order"]
    valid_qids = get_valid_dev_qids(candidates_artifact, qrels)
    normalized_fraction = normalize_dev_eval_fraction(fraction)
    if normalized_fraction >= 1.0:
        selected_qids = list(valid_qids)
    else:
        target_count = max(1, int(round(len(valid_qids) * normalized_fraction)))
        target_count = min(len(valid_qids), target_count)
        rng = random.Random(int(seed))
        sampled_qids = set(rng.sample(valid_qids, target_count))
        selected_qids = [int(qid) for qid in qid_order if int(qid) in sampled_qids and int(qid) in qrels]
    return build_dev_subset_for_qids(candidates_artifact, qrels, selected_qids)


def count_candidate_rows(candidates_artifact) -> int:
    return int(len(candidates_artifact["pid"]))


In [ ]:
seed_everything(SEED)

DATA_ROOT = Path(DATA_ROOT).expanduser()
OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser()
PREP_MANIFEST_PATH = Path(PREP_MANIFEST_PATH).expanduser()
prep_manifest = json.loads(PREP_MANIFEST_PATH.read_text())
validate_prep_manifest(prep_manifest)
BASE_ARTIFACT_DIR = PREP_MANIFEST_PATH.parent

train_triples_path = DATA_ROOT / TRAIN_TRIPLES_FILENAME
train_query_tokens_path = resolve_artifact_path(prep_manifest, "train_query_tokens_pkl", BASE_ARTIFACT_DIR)
dev_query_tokens_path = resolve_artifact_path(prep_manifest, "dev_query_tokens_pkl", BASE_ARTIFACT_DIR)
passage_token_shards_dir_path = resolve_artifact_path(prep_manifest, "passage_token_shards_dir", BASE_ARTIFACT_DIR, aliases=["passage_tokens_shards_dir"])
passage_token_shards_index_path = resolve_artifact_path(prep_manifest, "passage_token_shards_index_json", BASE_ARTIFACT_DIR, aliases=["passage_tokens_store_index_json"])
passage_token_store_stats_path = resolve_artifact_path(prep_manifest, "passage_token_store_stats_json", BASE_ARTIFACT_DIR, aliases=["passage_tokens_store_stats_json"])
dev_candidates_path = resolve_artifact_path(prep_manifest, "dev_candidates_pkl", BASE_ARTIFACT_DIR)
dev_qrels_path = resolve_artifact_path(prep_manifest, "dev_qrels_pkl", BASE_ARTIFACT_DIR)

base_cache_tag = str(prep_manifest.get("cache_tag") or BASE_ARTIFACT_DIR.name)
epoch_dev_mode_label = format_dev_eval_mode_label(
    DEV_EVAL_MODE,
    fraction=DEV_EVAL_FRACTION if DEV_EVAL_MODE == "quick_fraction" else None,
    query_count=DEV_EVAL_QUERY_COUNT if DEV_EVAL_MODE == "quick_count" else None,
)
RUN_DATA_CACHE_NAME = (
    f"{sanitize_tag_fragment(base_cache_tag)}_"
    f"sample{TRAIN_TRIPLES_SAMPLE}_seed{SEED}_dev{sanitize_tag_fragment(epoch_dev_mode_label)}_seed{DEV_EVAL_SEED}"
)
RUN_DATA_CACHE_DIR = OUTPUT_ROOT / "trm_reranker_mvp" / "run_data_cache" / RUN_DATA_CACHE_NAME
RUN_DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

sampled_train_triples_path = RUN_DATA_CACHE_DIR / f"train_triples_sample{TRAIN_TRIPLES_SAMPLE}_seed{SEED}.tsv"
train_query_tokens_output_path = RUN_DATA_CACHE_DIR / "train_query_tokens.pkl"
train_passage_tokens_output_path = RUN_DATA_CACHE_DIR / "train_passage_tokens.pkl"
epoch_dev_candidates_output_path = RUN_DATA_CACHE_DIR / "epoch_dev_candidates.pkl"
epoch_dev_qrels_output_path = RUN_DATA_CACHE_DIR / "epoch_dev_qrels.pkl"
run_data_manifest_path = RUN_DATA_CACHE_DIR / "run_data_manifest.json"

required_local_outputs = [
    sampled_train_triples_path,
    train_query_tokens_output_path,
    train_passage_tokens_output_path,
    epoch_dev_candidates_output_path,
    epoch_dev_qrels_output_path,
]

existing_manifest = None
if run_data_manifest_path.exists():
    existing_manifest = json.loads(run_data_manifest_path.read_text())

should_rebuild = FORCE_REBUILD_RUN_DATA or any(not path.exists() for path in required_local_outputs)
if existing_manifest is not None and not should_rebuild:
    compatibility_checks = {
        "train_triples_sample": int(existing_manifest.get("train_triples_sample", -1)) == int(TRAIN_TRIPLES_SAMPLE),
        "seed": int(existing_manifest.get("seed", -1)) == int(SEED),
        "dev_eval_mode": str(existing_manifest.get("dev_eval_mode")) == str(DEV_EVAL_MODE),
        "dev_eval_seed": int(existing_manifest.get("dev_eval_seed", -1)) == int(DEV_EVAL_SEED),
        "epoch_dev_mode_label": str(existing_manifest.get("epoch_dev_mode_label")) == str(epoch_dev_mode_label),
        "run_final_full_dev": bool(existing_manifest.get("run_final_full_dev", True)) == bool(RUN_FINAL_FULL_DEV),
    }
    if DEV_EVAL_MODE == "quick_count":
        compatibility_checks["dev_eval_query_count"] = int(existing_manifest.get("dev_eval_query_count", -1)) == normalize_dev_eval_query_count(DEV_EVAL_QUERY_COUNT)
    if DEV_EVAL_MODE == "quick_fraction":
        compatibility_checks["dev_eval_fraction"] = abs(float(existing_manifest.get("dev_eval_fraction", 0.0)) - normalize_dev_eval_fraction(DEV_EVAL_FRACTION)) < 1e-12
    if not all(compatibility_checks.values()):
        should_rebuild = True

if should_rebuild:
    full_train_query_tokens = load_pickle(train_query_tokens_path)
    dev_query_tokens = load_pickle(dev_query_tokens_path)
    dev_candidates = load_pickle(dev_candidates_path)
    dev_qrels = load_pickle(dev_qrels_path)
    passage_token_store_index, _get_cached_passage_tokens, load_passage_tokens_for_subset = build_passage_token_subset_loader(
        passage_token_shards_index_path,
        BASE_ARTIFACT_DIR,
        shards_dir_path=passage_token_shards_dir_path,
    )
    sampled_train_triples = build_run_train_subset(
        train_triples_path,
        sampled_train_triples_path,
        TRAIN_TRIPLES_SAMPLE,
        SEED,
        force=should_rebuild,
    )
    sampled_train_qids, sampled_train_pids = collect_train_subset_ids(sampled_train_triples)
    train_query_tokens = select_query_token_subset(full_train_query_tokens, sampled_train_qids, desc="Select train query tokens")
    train_passage_tokens = load_passage_tokens_for_subset(sampled_train_pids)
    save_pickle_highest_protocol(train_query_tokens, train_query_tokens_output_path)
    save_pickle_highest_protocol(train_passage_tokens, train_passage_tokens_output_path)

    if DEV_EVAL_MODE == "full":
        epoch_dev_candidates = dev_candidates
        epoch_dev_qrels = dev_qrels
    elif DEV_EVAL_MODE == "quick_count":
        epoch_dev_candidates, epoch_dev_qrels = build_count_dev_subset(
            dev_candidates,
            dev_qrels,
            query_count=DEV_EVAL_QUERY_COUNT,
            seed=DEV_EVAL_SEED,
        )
    elif DEV_EVAL_MODE == "quick_fraction":
        epoch_dev_candidates, epoch_dev_qrels = build_fractional_dev_subset(
            dev_candidates,
            dev_qrels,
            fraction=DEV_EVAL_FRACTION,
            seed=DEV_EVAL_SEED,
        )
    else:
        raise ValueError(f"Unsupported DEV_EVAL_MODE={DEV_EVAL_MODE!r}")

    save_pickle_highest_protocol(epoch_dev_candidates, epoch_dev_candidates_output_path)
    save_pickle_highest_protocol(epoch_dev_qrels, epoch_dev_qrels_output_path)

    run_data_manifest = {
        "schema_version": 1,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "run_data_scope": "run_level",
        "base_prep_manifest_path": maybe_relative_to(PREP_MANIFEST_PATH, RUN_DATA_CACHE_DIR),
        "base_artifact_dir": maybe_relative_to(BASE_ARTIFACT_DIR, RUN_DATA_CACHE_DIR),
        "base_cache_tag": base_cache_tag,
        "tokenizer_name": str(prep_manifest.get("tokenizer_name")),
        "seq_len": int(prep_manifest["seq_len"]),
        "max_query_len": int(prep_manifest["max_query_len"]),
        "max_doc_len": int(prep_manifest["max_doc_len"]),
        "train_triples_sample": int(TRAIN_TRIPLES_SAMPLE),
        "seed": int(SEED),
        "dev_eval_mode": DEV_EVAL_MODE,
        "epoch_dev_mode_label": epoch_dev_mode_label,
        "dev_eval_fraction": normalize_dev_eval_fraction(DEV_EVAL_FRACTION) if DEV_EVAL_MODE == "quick_fraction" else None,
        "dev_eval_query_count": normalize_dev_eval_query_count(DEV_EVAL_QUERY_COUNT) if DEV_EVAL_MODE == "quick_count" else None,
        "dev_eval_seed": int(DEV_EVAL_SEED),
        "run_final_full_dev": bool(RUN_FINAL_FULL_DEV),
        "counts": {
            "sampled_train_triples": int(len(sampled_train_triples)),
            "sampled_train_queries": int(len(sampled_train_qids)),
            "sampled_train_passages": int(len(sampled_train_pids)),
            "epoch_dev_queries": int(len(epoch_dev_candidates["qid_order"])),
            "epoch_dev_candidate_rows": int(count_candidate_rows(epoch_dev_candidates)),
        },
        "artifacts": {
            "sampled_train_triples_tsv": maybe_relative_to(sampled_train_triples_path, RUN_DATA_CACHE_DIR),
            "train_query_tokens_pkl": maybe_relative_to(train_query_tokens_output_path, RUN_DATA_CACHE_DIR),
            "train_passage_tokens_pkl": maybe_relative_to(train_passage_tokens_output_path, RUN_DATA_CACHE_DIR),
            "dev_query_tokens_pkl": maybe_relative_to(dev_query_tokens_path, BASE_ARTIFACT_DIR),
            "epoch_dev_candidates_pkl": maybe_relative_to(epoch_dev_candidates_output_path, RUN_DATA_CACHE_DIR),
            "epoch_dev_qrels_pkl": maybe_relative_to(epoch_dev_qrels_output_path, RUN_DATA_CACHE_DIR),
            "final_dev_candidates_pkl": maybe_relative_to(dev_candidates_path, BASE_ARTIFACT_DIR),
            "final_dev_qrels_pkl": maybe_relative_to(dev_qrels_path, BASE_ARTIFACT_DIR),
            "passage_token_shards_dir": maybe_relative_to(passage_token_shards_dir_path, BASE_ARTIFACT_DIR),
            "passage_token_shards_index_json": maybe_relative_to(passage_token_shards_index_path, BASE_ARTIFACT_DIR),
            "passage_token_store_stats_json": maybe_relative_to(passage_token_store_stats_path, BASE_ARTIFACT_DIR),
        },
        "passage_token_store_info": {
            "format": passage_token_store_index.get("format"),
            "shard_count": int(passage_token_store_index.get("shard_count", len(passage_token_store_index.get("shards", [])))),
        },
    }
    run_data_manifest_path.write_text(json.dumps(run_data_manifest, indent=2), encoding="utf-8")
else:
    run_data_manifest = existing_manifest

summary = {
    "run_data_manifest_path": str(run_data_manifest_path.resolve()),
    "run_data_cache_dir": str(RUN_DATA_CACHE_DIR.resolve()),
    "sampled_train_triples": int(run_data_manifest["counts"]["sampled_train_triples"]),
    "sampled_train_queries": int(run_data_manifest["counts"]["sampled_train_queries"]),
    "sampled_train_passages": int(run_data_manifest["counts"]["sampled_train_passages"]),
    "epoch_dev_queries": int(run_data_manifest["counts"]["epoch_dev_queries"]),
    "epoch_dev_candidate_rows": int(run_data_manifest["counts"]["epoch_dev_candidate_rows"]),
    "dev_eval_mode": run_data_manifest["dev_eval_mode"],
    "dev_eval_fraction": run_data_manifest.get("dev_eval_fraction"),
    "dev_eval_query_count": run_data_manifest.get("dev_eval_query_count"),
    "dev_eval_seed": run_data_manifest.get("dev_eval_seed"),
    "epoch_dev_mode_label": run_data_manifest["epoch_dev_mode_label"],
}
print(json.dumps(summary, indent=2))
summary
